# AlphaTrain Pillar 2b: Pairwise Ranked Value Head (Colab)

Warm start from Pillar 2a checkpoint (policy=494 mean). Fine-tune with pairwise
ranking loss to teach the value head relative move quality.

Upload `alphatrain_colab_td.tar.gz` to Google Drive first.
Also upload the Pillar 2a checkpoint as `alphatrain_2a_best.pt` to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/drive/MyDrive/alphatrain_colab_td.tar.gz /content/
!cd /content && tar xzf alphatrain_colab_td.tar.gz
!pip install -q numpy numba pytest scipy

In [ ]:
import torch, os
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Data: {os.path.getsize("/content/data/alphatrain_pairwise.pt") / 1e6:.0f} MB')
!cd /content && python -m pytest alphatrain/tests/ -v --tb=short

In [ ]:
%cd /content
# Warm start from Pillar 2a (policy=494), fine-tune with ranking loss
# --warm-start: loads weights but resets optimizer/scheduler
# Margin: tournament score diff scaled so mean=5 on value scale
# Watch: pol_loss should stay ~1.7-1.9 (not spike), rank_loss should stay >0.1
!python -m alphatrain.train \
    --tensor-file data/alphatrain_pairwise.pt \
    --gpu-data --amp --compile \
    --resume alphatrain/data/alphatrain_2a_best.pt --warm-start \
    --epochs 20 --batch-size 4096 --lr 3e-4 --warmup-epochs 2 \
    --val-weight 0.5 --rank-weight 1.0 \
    --copy-to /content/drive/MyDrive/alphatrain_td_best.pt \
    --save-dir /content/checkpoints/alphatrain_td

In [ ]:
!cp /content/checkpoints/alphatrain_td/latest.pt /content/drive/MyDrive/alphatrain_td_latest.pt
print('Saved to Drive')

In [ ]:
import sys
sys.path.insert(0, '/content')

# Evaluate standalone policy
!cd /content && python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/alphatrain_td/best.pt \
    --games 20 --seed 42

# Diagnose value head quality (key metric: rank correlation)
!cd /content && python -m alphatrain.scripts.debug_value_head \
    --model /content/checkpoints/alphatrain_td/best.pt --seed 42